# Dynasty Prospects — Colab Runner

Thin execution shell. All logic lives in the `dynasty_prospects` package in this repo — edit that in Claude Code, not here.

This notebook: clones the repo fresh → installs it → authenticates to GCP → runs the pipeline → writes to BigQuery.

## 1. Clone the repo

Set `REPO_URL` once. If the repo is private, add a `GITHUB_TOKEN` secret in Colab (key icon, left sidebar) first.

In [ ]:
REPO_URL = "https://github.com/nashstallings/dynasty-prospects.git"  # update this
REPO_DIR = "dynasty-prospects"

import os

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone {clone_url} {REPO_DIR}

## 2. Install the package

In [ ]:
%cd {REPO_DIR}
!pip install -q -e .
# nfl_data_py pins pandas<2.0/numpy<2.0, which would force pip to downgrade
# pandas project-wide and fail to build on Colab's Python version. It runs
# fine against the newer pandas installed above, so install it with --no-deps.
!pip install -q --no-deps nfl_data_py==0.3.3
%cd ..

## 3. Auth

GCP auth + CFBD API key. Get a free CFBD key at [collegefootballdata.com/key](https://collegefootballdata.com/key).

In [ ]:
from google.colab import auth
auth.authenticate_user()

import getpass
CFBD_API_KEY = getpass.getpass("Enter your CFBD API key: ")

## 4. Run the pipeline

In [ ]:
import sys
sys.path.insert(0, f"/content/{REPO_DIR}/src")

import dynasty_prospects

tables = dynasty_prospects.run(cfbd_api_key=CFBD_API_KEY, write_to_bq=True)
for name, df in tables.items():
    print(name, df.shape)

## 5. Verify

In [ ]:
from dynasty_prospects.bigquery_io import verify
from dynasty_prospects import config

verify(config.PROJECT_ID, config.DATASET_ID)

## 6. Scouting rankings snapshot (manual, run as needed)

Upload a filled-in CSV (columns: `player_name, position, school, source, snapshot_date, rank, tier_grade`). This pulls whatever's already in `fact_scouting_rankings` (if anything), appends your new dated snapshot, and writes the combined history back. The main pipeline (step 4) never touches this table, so re-running it won't wipe your scouting history.

In [ ]:
from google.colab import files
from dynasty_prospects import scouting, config
from dynasty_prospects.bigquery_io import read_table, write_tables

uploaded = files.upload()
new_path = list(uploaded.keys())[0]

existing_scouting = read_table(config.PROJECT_ID, config.DATASET_ID, "fact_scouting_rankings")
if existing_scouting is None:
    existing_scouting = scouting.empty_schema()

updated_scouting = scouting.append_snapshot(existing_scouting, new_path)
write_tables({"fact_scouting_rankings": updated_scouting}, config.PROJECT_ID, config.DATASET_ID)
updated_scouting.tail()